# GlucoCare AI - Exploratory Data Analysis
Educational notebook for the Pima Indians Diabetes dataset. Run it from the `notebooks/` folder after placing `diabetes.csv` in `data/`.
> Educational prototype - not a medical diagnostic system.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # lets us import the project's src package

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.preprocessing import (load_dataset, data_quality_report, zeros_to_nan,
                               FEATURE_COLUMNS, TARGET_COLUMN, ZERO_AS_MISSING_COLUMNS)
sns.set_theme(style="whitegrid", palette="GnBu_d")

## 1. Load and inspect

In [ ]:
df = load_dataset()
print(df.shape)
df.head()

In [ ]:
df.info()
df.describe().T

## 2. Missing values, duplicates and impossible zeros
A value of 0 for Glucose, BloodPressure, SkinThickness, Insulin or BMI is not physiologically possible - it really means *not recorded*.

In [ ]:
rep = data_quality_report(df)
print("NaN values:", sum(rep["missing_values"].values()))
print("Duplicate rows:", rep["duplicate_rows"])
pd.DataFrame({"zeros": rep["invalid_zero_counts"],
              "% of rows": {k: round(v/len(df)*100, 1) for k, v in rep["invalid_zero_counts"].items()}})

## 3. Target distribution

In [ ]:
counts = df[TARGET_COLUMN].value_counts().sort_index()
print(counts, "\n", (counts/len(df)*100).round(1))
sns.countplot(x=TARGET_COLUMN, data=df)
plt.xticks([0, 1], ["Non-diabetic", "Diabetic"]); plt.title("Class distribution"); plt.show()

## 4. Feature distributions (impossible zeros treated as missing)

In [ ]:
clean = zeros_to_nan(df[FEATURE_COLUMNS])
clean[TARGET_COLUMN] = df[TARGET_COLUMN].values
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), FEATURE_COLUMNS):
    sns.histplot(data=clean, x=col, hue=TARGET_COLUMN, kde=True, bins=25, ax=ax)
plt.tight_layout(); plt.show()

## 5. Outliers and class differences

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), FEATURE_COLUMNS):
    sns.boxplot(data=clean, x=TARGET_COLUMN, y=col, ax=ax)
plt.tight_layout(); plt.show()

## 6. Correlation analysis

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(clean.corr(), annot=True, fmt=".2f", cmap="GnBu", square=True)
plt.title("Correlation heatmap (zeros treated as missing)"); plt.show()
clean.corr()[TARGET_COLUMN].drop(TARGET_COLUMN).sort_values(ascending=False)

## 7. Observations (fill in after running)
- Which features are most correlated with the outcome?
- Which columns have the most hidden missing values, and why does that matter?
- Is the dataset balanced? Which metrics are therefore preferable to accuracy?